In [1]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
def load_labeled_data(data_dir):
    texts = []
    labels = []
    for label_name, label_idx in [('pos', 1), ('neg', 0)]:
        folder_path = os.path.join(data_dir, label_name)
        if not os.path.exists(folder_path): continue
        
        files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
        print(f"Loading {label_name} files from {data_dir}...")
        for fname in tqdm(files, desc=label_name):
            fpath = os.path.join(folder_path, fname)
            with open(fpath, 'r', encoding='utf-8') as f:
                text = f.read().strip()
                if text:
                    texts.append(text)
                    labels.append(label_idx)
    return texts, labels

train_dir = r"d:\dut_ai\AIO_code\Dence Representation\data\data_train\train"
test_dir = r"d:\dut_ai\AIO_code\Dence Representation\data\data_test\test"

train_texts, train_labels = load_labeled_data(train_dir)
test_texts, test_labels = load_labeled_data(test_dir)

print(f"\nTotal Train source files: {len(train_texts)}, Test source files: {len(test_texts)}")

Loading pos files from d:\dut_ai\AIO_code\Dence Representation\data\data_train\train...


pos: 100%|██████████| 15000/15000 [02:45<00:00, 90.84it/s] 


Loading neg files from d:\dut_ai\AIO_code\Dence Representation\data\data_train\train...


neg: 100%|██████████| 15000/15000 [01:28<00:00, 169.59it/s]


Loading pos files from d:\dut_ai\AIO_code\Dence Representation\data\data_test\test...


pos: 100%|██████████| 5000/5000 [00:01<00:00, 3227.53it/s]


Loading neg files from d:\dut_ai\AIO_code\Dence Representation\data\data_test\test...


neg: 100%|██████████| 5000/5000 [00:04<00:00, 1183.83it/s]


Total Train source files: 30000, Test source files: 10000


In [3]:
def tokenize(text):
    text = text.lower()
    return re.findall(r'[a-záàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ_]+', text)

print("Xây dựng từ điển từ tập Train...")
all_tokens = []
for text in tqdm(train_texts, desc="Tokenizing"):
    all_tokens.extend(tokenize(text))

min_freq = 3
freq = Counter(all_tokens)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in freq.items() if c >= min_freq]
word2idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

Xây dựng từ điển từ tập Train...


Tokenizing: 100%|██████████| 30000/30000 [00:01<00:00, 19815.51it/s]


Vocab size: 13677


In [4]:
MAX_LEN = 50

def texts_to_sequences(texts, word2idx, max_len):
    sequences = []
    for text in tqdm(texts, desc="Processing sequences"):
        tokens = tokenize(text)
        indices = [word2idx.get(w, word2idx['<UNK>']) for w in tokens]
        if len(indices) < max_len:
            indices = indices + [word2idx['<PAD>']] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        sequences.append(indices)
    return np.array(sequences)

X_train_full = texts_to_sequences(train_texts, word2idx, MAX_LEN)
y_train_full = np.array(train_labels)

X_test = torch.tensor(texts_to_sequences(test_texts, word2idx, MAX_LEN), dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)

# Chia Train -> Train (80%) và Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val = torch.tensor(X_val, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Processing sequences: 100%|██████████| 10000/10000 [00:00<00:00, 19024.12it/s]


Train: torch.Size([24000, 50]), Val: torch.Size([6000, 50]), Test: torch.Size([10000, 50])


In [5]:
class SentimentMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(
            nn.Linear(max_len * embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = self.embedding(x)      
        x = self.flatten(x)        
        return self.mlp(x)         

EMBED_DIM = 100
model = SentimentMLP(vocab_size, EMBED_DIM, MAX_LEN)

In [6]:
EPOCHS = 10
LR = 0.001
BATCH_SIZE = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_ds = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = torch.utils.data.TensorDataset(X_val, y_val)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE)

for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * xb.size(0)
        preds = (out > 0.5).float()
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total

    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            preds = (out > 0.5).float()
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)
            
    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}\n")

Epoch 1/10 [Train]: 100%|██████████| 188/188 [00:07<00:00, 23.59it/s, loss=0.5034]


Epoch 1: Train Loss: 0.5878, Train Acc: 0.6777 | Val Loss: 0.5567, Val Acc: 0.7152



Epoch 2/10 [Train]: 100%|██████████| 188/188 [00:07<00:00, 24.29it/s, loss=0.3213]


Epoch 2: Train Loss: 0.3720, Train Acc: 0.8353 | Val Loss: 0.5187, Val Acc: 0.7698



Epoch 3/10 [Train]: 100%|██████████| 188/188 [00:07<00:00, 23.57it/s, loss=0.1941]


...

Epoch 8: Train Loss: 0.0504, Train Acc: 0.9807 | Val Loss: 1.5817, Val Acc: 0.7630



Epoch 9/10 [Train]: 100%|██████████| 188/188 [00:07<00:00, 24.52it/s, loss=0.0459]


Epoch 9: Train Loss: 0.0418, Train Acc: 0.9848 | Val Loss: 1.6495, Val Acc: 0.7650



Epoch 10/10 [Train]: 100%|██████████| 188/188 [00:07<00:00, 24.70it/s, loss=0.0317]


Epoch 10: Train Loss: 0.0404, Train Acc: 0.9851 | Val Loss: 1.8563, Val Acc: 0.7628



In [7]:
print("Đánh giá trên tập TEST thực tế (unseen data)...")
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for xb, yb in tqdm(torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_test, y_test), batch_size=BATCH_SIZE), desc="Evaluating Test"):
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        preds = (out > 0.5).float()
        test_correct += (preds == yb).sum().item()
        test_total += yb.size(0)

print(f"\nTest Accuracy: {test_correct/test_total:.4f}")

Đánh giá trên tập TEST thực tế (unseen data)...


Evaluating Test: 100%|██████████| 79/79 [00:00<00:00, 163.87it/s]


Test Accuracy: 0.7657


In [8]:
def predict_sentiment(text, model, word2idx, max_len):
    model.eval()
    tokens = tokenize(text)
    indices = [word2idx.get(w, word2idx['<UNK>']) for w in tokens]
    if len(indices) < max_len:
        indices = indices + [word2idx['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    
    input_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    with torch.no_grad():
        prob = model(input_tensor).item()
    
    sentiment = "Tích cực (POS)" if prob > 0.5 else "Tiêu cực (NEG)"
    print(f"Câu: {text}")
    print(f"Dự đoán: {sentiment} ({prob*100:.2f}%)")

predict_sentiment("Món ăn rất ngon, phục vụ tận tình", model, word2idx, MAX_LEN)
predict_sentiment("Quá tệ, tôi sẽ không bao giờ quay lại", model, word2idx, MAX_LEN)

Câu: Món ăn rất ngon, phục vụ tận tình
Dự đoán: Tiêu cực (NEG) (26.36%)
Câu: Quá tệ, tôi sẽ không bao giờ quay lại
Dự đoán: Tiêu cực (NEG) (0.03%)
